# Lab 1：Tushare 金融数据质量验证


本实验把数据源当作需要验收的研究基础设施，而不是把“API 返回 200”当作成功。当前结论将在执行后由底部汇总单元给出；判断标准包括：

- 行情是否覆盖预期交易日、是否重复、OHLC 与金额字段是否合法；
- `daily` 是否确为不复权价格，以及 `adj_factor` 能否独立重建前复权价格；
- 分红金额是否为每股金额、税前税后字段如何区分、各日期分别代表什么；
- 股票与 ETF 是否共用接口、字段与单位是否一致；
- AkShare 与 Tushare 在同一标的同一日期上的价格、成交量和分红是否一致；
- 数据是否发生历史修订（首次运行建立基线，后续运行自动比较并归档旧版）。

主要产物写入 `labs/data/lab1_tushare/`：银行股清单、不复权日行情、前复权校验行情、分红原始数据、数据字段说明、缺失数据报告、数据源对照表。

## Context & Methods

### 研究范围

- A 股银行清单：Tushare `stock_basic` 中当前上市且 `industry == "银行"` 的股票。
- 样本股票：工商银行、建设银行、农业银行、中国银行、招商银行。
- 样本 ETF：沪深300ETF（`510300.SH`）。
- 默认区间：2022-01-01 至运行日；可通过 `LAB1_START_DATE` / `LAB1_END_DATE` 环境变量覆盖。
- 数据粒度：日行情为 `证券 × 交易日`；分红为 `证券 × 分红年度 × 公告/实施阶段`。

### 关键假设

1. Tushare `daily` 是未复权行情，停牌日不返回记录；`vol` 单位为手，`amount` 单位为千元。
2. 股票前复权价按 `raw_price × 当日复权因子 ÷ 区间最新复权因子` 重建。
3. Tushare 分红 `cash_div` / `cash_div_tax` 是每股税后/税前人民币金额；AkShare 的“现金分红比例”按“每10股派 X 元”解释，比较前除以 10。
4. 历史修订无法由一次抓取证明不存在。本 notebook 将本次数据与上次本地快照比较；首次运行仅建立基线。

### 官方字段文档

- [Tushare A股日线](https://tushare.pro/document/1?doc_id=27)
- [Tushare 股票复权因子](https://tushare.pro/document/2?doc_id=28)
- [Tushare 分红送股](https://tushare.pro/document/2?doc_id=103)
- [Tushare ETF 日线](https://tushare.pro/document/2?doc_id=127)
- [Tushare ETF 复权因子](https://tushare.pro/document/2?doc_id=199)
- [AkShare 股票数据文档](https://akshare.akfamily.xyz/data/stock/stock.html)

## Data

In [1]:
from __future__ import annotations

import hashlib
import inspect
import json
import os
import shutil
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests
import tushare as ts
from IPython.display import Markdown, display
from dotenv import load_dotenv

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)

# 无论从仓库根目录还是 labs/ 启动，都定位到含 .env 的项目根目录。
PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
load_dotenv(PROJECT_ROOT / ".env")

DATA_DIR = PROJECT_ROOT / "labs" / "data" / "lab1_tushare"
ARCHIVE_DIR = DATA_DIR / "archive"
DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = os.getenv("LAB1_START_DATE", "20220101")
END_DATE = os.getenv("LAB1_END_DATE", pd.Timestamp.now(tz="Asia/Shanghai").strftime("%Y%m%d"))
RUN_AT = pd.Timestamp.now(tz="Asia/Shanghai")
RUN_STAMP = RUN_AT.strftime("%Y%m%dT%H%M%S%z")

TARGET_STOCKS = {
    "601398.SH": "工商银行",
    "601939.SH": "建设银行",
    "601288.SH": "农业银行",
    "601988.SH": "中国银行",
    "600036.SH": "招商银行",
}
REPRESENTATIVE_STOCK = "600036.SH"
REPRESENTATIVE_ETF = "510300.SH"

token = os.getenv("TUSHARE_APIKEY", "").strip()
if not token:
    raise RuntimeError(".env 中缺少 TUSHARE_APIKEY；notebook 不会打印或持久化该密钥。")

# 只让 Tushare 绕过系统代理：Tushare 1.4.25 的 DataApi.query 使用
# tushare.pro.client.requests.post。替换该模块引用为专用 Session，避免影响
# AkShare 和其他 requests 调用，它们仍按系统/环境代理设置工作。
import tushare.pro.client as tushare_client

TUSHARE_DIRECT_SESSION = requests.Session()
TUSHARE_DIRECT_SESSION.trust_env = False
tushare_client.requests = TUSHARE_DIRECT_SESSION
pro = ts.pro_api(token)

proxy_presence = {
    key: bool(os.getenv(key))
    for key in ("HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY", "NO_PROXY")
}
print({
    "tushare_version": ts.__version__,
    "date_range": f"{START_DATE}..{END_DATE}",
    "tushare_trust_env": TUSHARE_DIRECT_SESSION.trust_env,
    "proxy_variables_present_without_values": proxy_presence,
    "output_dir": str(DATA_DIR.relative_to(PROJECT_ROOT)),
})

{'tushare_version': '1.4.25', 'date_range': '20220101..20260723', 'tushare_trust_env': False, 'proxy_variables_present_without_values': {'HTTP_PROXY': False, 'HTTPS_PROXY': False, 'ALL_PROXY': False, 'NO_PROXY': False}, 'output_dir': 'labs\\data\\lab1_tushare'}


In [2]:
revision_findings: list[dict] = []


def tushare_query(method, *, label: str, retries: int = 3, **kwargs) -> pd.DataFrame:
    """串行、带退避的 Tushare 调用；错误保留接口标签但不泄露 token。"""
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            frame = method(**kwargs)
            if not isinstance(frame, pd.DataFrame):
                raise TypeError(f"返回类型不是 DataFrame: {type(frame)!r}")
            print(f"[OK] {label}: {len(frame):,} rows")
            return frame
        except Exception as exc:
            last_error = exc
            if attempt < retries:
                time.sleep(attempt * 1.5)
    raise RuntimeError(f"{label} 调用失败: {type(last_error).__name__}: {last_error}") from last_error


def stable_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """将 DataFrame 归一化为适合跨运行比较的字符串表。"""
    out = frame.copy()
    out = out.reindex(sorted(out.columns), axis=1)
    for col in out.columns:
        # CSV 会把缺失值写成空字段；统一为空串，避免“NaN vs 空串”造成假修订。
        out[col] = out[col].where(out[col].notna(), "").astype(str)
    if len(out.columns):
        out = out.sort_values(list(out.columns), kind="stable").reset_index(drop=True)
    return out


def save_dataset(frame: pd.DataFrame, filename: str, keys: list[str] | None = None) -> Path:
    """保存当前版；若与上次不同，归档旧版并记录增删/同键变更。"""
    path = DATA_DIR / filename
    current = frame.copy()
    current_stable = stable_frame(current)
    current_hash = hashlib.sha256(
        current_stable.to_csv(index=False, lineterminator="\n").encode("utf-8")
    ).hexdigest()

    finding = {
        "dataset": filename,
        "run_at": RUN_AT.isoformat(),
        "status": "baseline",
        "previous_rows": np.nan,
        "current_rows": len(current),
        "added_exact_rows": np.nan,
        "removed_exact_rows": np.nan,
        "changed_keys": np.nan,
        "current_sha256": current_hash,
        "note": "首次运行或无旧快照，尚不能判断历史修订",
    }

    if path.exists():
        previous = pd.read_csv(path, dtype=str, keep_default_na=False)
        previous_stable = stable_frame(previous)
        previous_hash = hashlib.sha256(
            previous_stable.to_csv(index=False, lineterminator="\n").encode("utf-8")
        ).hexdigest()
        finding["previous_rows"] = len(previous)
        if previous_hash == current_hash:
            finding.update({
                "status": "unchanged",
                "added_exact_rows": 0,
                "removed_exact_rows": 0,
                "changed_keys": 0,
                "note": "与上次本地快照完全一致",
            })
        else:
            old_rows = set(previous_stable.astype(str).agg("\x1f".join, axis=1))
            new_rows = set(current_stable.astype(str).agg("\x1f".join, axis=1))
            changed_keys = np.nan
            usable_keys = [k for k in (keys or []) if k in previous.columns and k in current.columns]
            if usable_keys:
                old_keyed = previous.drop_duplicates(usable_keys, keep="last").set_index(usable_keys)
                new_keyed = current.astype(str).drop_duplicates(usable_keys, keep="last").set_index(usable_keys)
                common = old_keyed.index.intersection(new_keyed.index)
                common_cols = [c for c in old_keyed.columns if c in new_keyed.columns]
                changed_keys = 0
                if len(common) and common_cols:
                    left = old_keyed.loc[common, common_cols].fillna("").astype(str)
                    right = new_keyed.loc[common, common_cols].fillna("").astype(str)
                    changed_keys = int(left.ne(right).any(axis=1).sum())

            archive_run = ARCHIVE_DIR / RUN_STAMP
            archive_run.mkdir(parents=True, exist_ok=True)
            shutil.copy2(path, archive_run / filename)
            finding.update({
                "status": "revised",
                "added_exact_rows": len(new_rows - old_rows),
                "removed_exact_rows": len(old_rows - new_rows),
                "changed_keys": changed_keys,
                "note": f"旧版已归档到 {archive_run.relative_to(PROJECT_ROOT)}",
            })

    current.to_csv(path, index=False, encoding="utf-8-sig")
    revision_findings.append(finding)
    return path


def sort_daily(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.sort_values(["ts_code", "trade_date"], ascending=[True, True]).reset_index(drop=True)


PRICE_COLUMNS = ["open", "high", "low", "close", "pre_close"]


def add_qfq(raw: pd.DataFrame, factors: pd.DataFrame) -> pd.DataFrame:
    """按每只证券的最新因子归一化，独立重建前复权价格。"""
    merged = raw.merge(factors, on=["ts_code", "trade_date"], how="left", validate="one_to_one")
    merged["latest_adj_factor"] = merged.groupby("ts_code")["adj_factor"].transform("last")
    for col in PRICE_COLUMNS:
        if col in merged.columns:
            merged[f"{col}_qfq"] = merged[col] * merged["adj_factor"] / merged["latest_adj_factor"]
    return sort_daily(merged)

### 1. 银行股清单

In [3]:
stock_basic = tushare_query(
    pro.stock_basic,
    label="stock_basic 当前上市股票",
    exchange="",
    list_status="L",
    fields="ts_code,symbol,name,area,industry,market,exchange,list_status,list_date",
)
bank_list = (
    stock_basic.loc[stock_basic["industry"].eq("银行")]
    .sort_values("ts_code")
    .reset_index(drop=True)
)
bank_list.insert(0, "序号", np.arange(1, len(bank_list) + 1))

ak_bank_path = PROJECT_ROOT / "labs" / "data" / "lab1_akshare" / "银行板块清单.csv"
if ak_bank_path.exists():
    ak_bank_list_cached = pd.read_csv(ak_bank_path, dtype={"代码": str})
    ak_bank_codes = set(ak_bank_list_cached["代码"].astype(str).str.zfill(6))
else:
    ak_bank_list_cached = pd.DataFrame()
    ak_bank_codes = set()

ts_bank_codes = set(bank_list["symbol"])
bank_membership_check = pd.DataFrame({
    "metric": ["Tushare银行数", "AkShare缓存银行数", "仅Tushare", "仅AkShare"],
    "value": [len(ts_bank_codes), len(ak_bank_codes), len(ts_bank_codes-ak_bank_codes), len(ak_bank_codes-ts_bank_codes)],
    "detail": ["", "", ",".join(sorted(ts_bank_codes-ak_bank_codes)), ",".join(sorted(ak_bank_codes-ts_bank_codes))],
})
save_dataset(bank_list, "银行股清单.csv", keys=["ts_code"])
display(bank_membership_check)
display(bank_list.head(10))

[OK] stock_basic 当前上市股票: 5,530 rows


,metric,value,detail
0,Tushare银行数,42,
1,AkShare缓存银行数,42,
2,仅Tushare,0,
3,仅AkShare,0,


,序号,ts_code,symbol,name,area,industry,market,exchange,list_status,list_date
0,1,000001.SZ,000001,平安银行,深圳,银行,主板,SZSE,L,19910403
1,2,001227.SZ,001227,兰州银行,甘肃,银行,主板,SZSE,L,20220117
2,3,002142.SZ,002142,宁波银行,浙江,银行,主板,SZSE,L,20070719
3,4,002807.SZ,002807,江阴银行,江苏,银行,主板,SZSE,L,20160902
4,5,002839.SZ,002839,张家港行,江苏,银行,主板,SZSE,L,20170124
5,6,002936.SZ,002936,郑州银行,河南,银行,主板,SZSE,L,20180919
6,7,002948.SZ,002948,青岛银行,山东,银行,主板,SZSE,L,20190116
7,8,002958.SZ,002958,青农商行,山东,银行,主板,SZSE,L,20190326
8,9,002966.SZ,002966,苏州银行,江苏,银行,主板,SZSE,L,20190802
9,10,600000.SH,600000,浦发银行,上海,银行,主板,SSE,L,19991110


### 2. 不复权日行情与前复权重建

In [4]:
trade_calendar = tushare_query(
    pro.trade_cal,
    label="SSE交易日历",
    exchange="SSE",
    start_date=START_DATE,
    end_date=END_DATE,
    fields="exchange,cal_date,is_open,pretrade_date",
).sort_values("cal_date")

daily_parts = []
factor_parts = []
for ts_code, name in TARGET_STOCKS.items():
    daily = tushare_query(
        pro.daily,
        label=f"daily {name} {ts_code}",
        ts_code=ts_code,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    factors = tushare_query(
        pro.adj_factor,
        label=f"adj_factor {name} {ts_code}",
        ts_code=ts_code,
        start_date=START_DATE,
        end_date=END_DATE,
    )
    daily_parts.append(daily)
    factor_parts.append(factors)

stock_daily_raw = sort_daily(pd.concat(daily_parts, ignore_index=True))
stock_adj_factors = sort_daily(pd.concat(factor_parts, ignore_index=True))
stock_daily_qfq_check = add_qfq(stock_daily_raw, stock_adj_factors)

save_dataset(stock_daily_raw, "不复权日行情.csv", keys=["ts_code", "trade_date"])
save_dataset(stock_adj_factors, "股票复权因子.csv", keys=["ts_code", "trade_date"])
save_dataset(stock_daily_qfq_check, "前复权校验行情.csv", keys=["ts_code", "trade_date"])

display(stock_daily_raw.groupby("ts_code").agg(rows=("trade_date", "size"), start=("trade_date", "min"), end=("trade_date", "max")))
display(stock_daily_qfq_check.loc[stock_daily_qfq_check["ts_code"].eq(REPRESENTATIVE_STOCK)].tail())

[OK] SSE交易日历: 1,665 rows
[OK] daily 工商银行 601398.SH: 1,101 rows
[OK] adj_factor 工商银行 601398.SH: 1,102 rows
[OK] daily 建设银行 601939.SH: 1,101 rows
[OK] adj_factor 建设银行 601939.SH: 1,102 rows
[OK] daily 农业银行 601288.SH: 1,101 rows
[OK] adj_factor 农业银行 601288.SH: 1,102 rows
[OK] daily 中国银行 601988.SH: 1,101 rows
[OK] adj_factor 中国银行 601988.SH: 1,102 rows
[OK] daily 招商银行 600036.SH: 1,101 rows
[OK] adj_factor 招商银行 600036.SH: 1,102 rows


,rows,start,end
ts_code,,,
600036.SH,1101,20220104,20260722
601288.SH,1101,20220104,20260722
601398.SH,1101,20220104,20260722
601939.SH,1101,20220104,20260722
601988.SH,1101,20220104,20260722


,ts_code,trade_date,open,high,low,close,pre_close,change,pct_chg,vol,amount,adj_factor,latest_adj_factor,open_qfq,high_qfq,low_qfq,close_qfq,pre_close_qfq
1096,600036.SH,20260716,37.70,38.12,37.50,37.67,37.76,-0.09,-0.2383,766898.70,2894610.320,6.6608,6.6608,37.70,38.12,37.50,37.67,37.76
1097,600036.SH,20260717,37.76,38.20,37.68,38.02,37.67,0.35,0.9291,1158584.49,4404072.223,6.6608,6.6608,37.76,38.20,37.68,38.02,37.67
1098,600036.SH,20260720,37.95,38.91,37.89,38.91,38.02,0.89,2.3409,1433610.15,5534176.002,6.6608,6.6608,37.95,38.91,37.89,38.91,38.02
1099,600036.SH,20260721,39.18,39.40,37.90,37.99,38.91,-0.92,-2.3644,1560127.42,6018519.241,6.6608,6.6608,39.18,39.40,37.90,37.99,38.91
1100,600036.SH,20260722,37.85,38.71,37.63,38.71,37.99,0.72,1.8952,1084545.98,4146124.500,6.6608,6.6608,37.85,38.71,37.63,38.71,37.99


### 3. 分红原始数据与日期语义

In [5]:
DIVIDEND_FIELDS = (
    "ts_code,end_date,ann_date,div_proc,stk_div,stk_bo_rate,stk_co_rate,"
    "cash_div,cash_div_tax,record_date,ex_date,pay_date,div_listdate,"
    "imp_ann_date,base_date,base_share"
)
dividend_parts = []
for ts_code, name in TARGET_STOCKS.items():
    dividend_parts.append(tushare_query(
        pro.dividend,
        label=f"dividend {name} {ts_code}",
        ts_code=ts_code,
        fields=DIVIDEND_FIELDS,
    ))

dividends_raw = (
    pd.concat(dividend_parts, ignore_index=True)
    .sort_values(["ts_code", "end_date", "ann_date"], ascending=[True, False, False], na_position="last")
    .reset_index(drop=True)
)
save_dataset(
    dividends_raw,
    "分红原始数据.csv",
    keys=["ts_code", "end_date", "ann_date", "div_proc", "imp_ann_date"],
)

dividend_date_semantics = pd.DataFrame([
    ["end_date", "分红归属的财务年度/报告期", "不是公告日，也不是到账日"],
    ["ann_date", "预案公告日", "董事会/公司首次披露分红预案的日期"],
    ["imp_ann_date", "实施公告日", "权益分派实施细节正式披露日"],
    ["record_date", "股权登记日", "当日收盘后在册股东取得本次权益"],
    ["ex_date", "除权除息日", "当日买入者不再取得本次权益；价格基准相应调整"],
    ["pay_date", "派息日", "现金股利实际发放/到账日期"],
    ["div_listdate", "红股上市日", "送股/转增股份可上市交易日期"],
    ["base_date", "分派基准日", "计算分派基数所对应日期，可能为空"],
], columns=["field", "meaning", "caveat"])
display(dividend_date_semantics)
display(dividends_raw.loc[(dividends_raw["ts_code"].eq(REPRESENTATIVE_STOCK)) & (dividends_raw["div_proc"].eq("实施"))].head(8))

[OK] dividend 工商银行 601398.SH: 59 rows
[OK] dividend 建设银行 601939.SH: 48 rows
[OK] dividend 农业银行 601288.SH: 44 rows
[OK] dividend 中国银行 601988.SH: 48 rows
[OK] dividend 招商银行 600036.SH: 52 rows


,field,meaning,caveat
0,end_date,分红归属的财务年度/报告期,不是公告日，也不是到账日
1,ann_date,预案公告日,董事会/公司首次披露分红预案的日期
2,imp_ann_date,实施公告日,权益分派实施细节正式披露日
3,record_date,股权登记日,当日收盘后在册股东取得本次权益
4,ex_date,除权除息日,当日买入者不再取得本次权益；价格基准相应调整
5,pay_date,派息日,现金股利实际发放/到账日期
6,div_listdate,红股上市日,送股/转增股份可上市交易日期
7,base_date,分派基准日,计算分派基数所对应日期，可能为空


,ts_code,end_date,ann_date,div_proc,stk_div,stk_bo_rate,stk_co_rate,cash_div,cash_div_tax,record_date,ex_date,pay_date,div_listdate,imp_ann_date,base_date,base_share
2,600036.SH,20251231,20260626,实施,0.0,NaN,NaN,1.003,1.003,20260709,20260710,20260710,NaN,20260704,20260703,2.521985e+06
5,600036.SH,20250630,NaN,实施,0.0,NaN,NaN,1.013,1.013,20260115,20260116,20260116,NaN,20260110,20260109,2.521985e+06
8,600036.SH,20241231,20250326,实施,0.0,NaN,NaN,2.000,2.000,20250710,20250711,20250711,NaN,20250704,20250704,2.521985e+06
12,600036.SH,20231231,20240326,实施,0.0,NaN,NaN,1.972,1.972,20240710,20240711,20240711,NaN,20240704,20240704,2.521985e+06
16,600036.SH,20221231,20230325,实施,0.0,NaN,NaN,1.738,1.738,20230712,20230713,20230713,NaN,20230706,20230706,2.521985e+06
20,600036.SH,20211231,20220319,实施,0.0,NaN,NaN,1.522,1.522,20220714,20220715,20220715,NaN,20220708,20220708,2.521985e+06
24,600036.SH,20201231,20210320,实施,0.0,NaN,NaN,1.253,1.253,20210712,20210713,20210713,NaN,20210706,20210712,2.521985e+06
28,600036.SH,20191231,20200321,实施,0.0,NaN,NaN,1.200,1.200,20200709,20200710,20200710,NaN,20200703,20200709,2.521985e+06


### 4. ETF 与股票接口一致性

In [6]:
etf_daily_raw = tushare_query(
    pro.fund_daily,
    label=f"fund_daily {REPRESENTATIVE_ETF}",
    ts_code=REPRESENTATIVE_ETF,
    start_date=START_DATE,
    end_date=END_DATE,
)
etf_adj_factors = tushare_query(
    pro.fund_adj,
    label=f"fund_adj {REPRESENTATIVE_ETF}",
    ts_code=REPRESENTATIVE_ETF,
    start_date=START_DATE,
    end_date=END_DATE,
)
etf_daily_raw = sort_daily(etf_daily_raw)
etf_adj_factors = sort_daily(etf_adj_factors)
etf_daily_qfq_check = add_qfq(etf_daily_raw, etf_adj_factors)

save_dataset(etf_daily_raw, "ETF不复权日行情.csv", keys=["ts_code", "trade_date"])
save_dataset(etf_adj_factors, "ETF复权因子.csv", keys=["ts_code", "trade_date"])
save_dataset(etf_daily_qfq_check, "ETF前复权校验行情.csv", keys=["ts_code", "trade_date"])

stock_fields = set(stock_daily_raw.columns)
etf_fields = set(etf_daily_raw.columns)
stock_etf_schema_check = pd.DataFrame({
    "metric": ["股票接口", "ETF接口", "共同字段", "仅股票字段", "仅ETF字段"],
    "value": [
        "pro.daily / pro.adj_factor",
        "pro.fund_daily / pro.fund_adj",
        ", ".join(sorted(stock_fields & etf_fields)),
        ", ".join(sorted(stock_fields - etf_fields)),
        ", ".join(sorted(etf_fields - stock_fields)),
    ],
})
display(stock_etf_schema_check)

[OK] fund_daily 510300.SH: 1,101 rows
[OK] fund_adj 510300.SH: 1,101 rows


,metric,value
0,股票接口,pro.daily / pro.adj_factor
1,ETF接口,pro.fund_daily / pro.fund_adj
2,共同字段,"amount, change, close, high, low, open, pct_ch..."
3,仅股票字段,
4,仅ETF字段,


## Results

### 5. AkShare 交叉验证（保留系统代理）

In [7]:
akshare_results: dict[str, pd.DataFrame] = {}
akshare_errors: dict[str, str] = {}

try:
    import akshare as ak

    def ak_call(label, func, retries=2):
        last = None
        for attempt in range(retries):
            try:
                result = func()
                print(f"[OK] AkShare {label}: {len(result):,} rows")
                return result
            except Exception as exc:
                last = exc
                if attempt + 1 < retries:
                    time.sleep(2)
        akshare_errors[label] = f"{type(last).__name__}: {last}"
        print(f"[WARN] AkShare {label}: {akshare_errors[label]}")
        return pd.DataFrame()

    rep_symbol = REPRESENTATIVE_STOCK.split(".")[0]
    etf_symbol = REPRESENTATIVE_ETF.split(".")[0]
    akshare_results["stock_raw"] = ak_call("股票不复权", lambda: ak.stock_zh_a_hist(
        symbol=rep_symbol, period="daily", start_date=START_DATE, end_date=END_DATE, adjust=""))
    akshare_results["stock_qfq"] = ak_call("股票前复权", lambda: ak.stock_zh_a_hist(
        symbol=rep_symbol, period="daily", start_date=START_DATE, end_date=END_DATE, adjust="qfq"))
    akshare_results["dividend"] = ak_call("股票分红", lambda: ak.stock_fhps_detail_em(symbol=rep_symbol))
    akshare_results["etf_raw"] = ak_call("ETF不复权", lambda: ak.fund_etf_hist_em(
        symbol=etf_symbol, period="daily", start_date=START_DATE, end_date=END_DATE, adjust=""))
    akshare_results["etf_qfq"] = ak_call("ETF前复权", lambda: ak.fund_etf_hist_em(
        symbol=etf_symbol, period="daily", start_date=START_DATE, end_date=END_DATE, adjust="qfq"))
except ImportError as exc:
    akshare_errors["import"] = str(exc)

print("AkShare 默认 requests 行为未被修改；仅 Tushare 专用 Session 设置 trust_env=False。")

[OK] AkShare 股票不复权: 1,102 rows
[OK] AkShare 股票前复权: 1,102 rows


  0%|          | 0/1 [00:00<?, ?it/s]

[OK] AkShare 股票分红: 26 rows
[OK] AkShare ETF不复权: 1,102 rows
[OK] AkShare ETF前复权: 1,102 rows
AkShare 默认 requests 行为未被修改；仅 Tushare 专用 Session 设置 trust_env=False。


In [8]:
def normalize_ak_daily(frame: pd.DataFrame, ts_code: str) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    out = frame.rename(columns={
        "日期": "trade_date", "开盘": "open", "最高": "high", "最低": "low",
        "收盘": "close", "成交量": "vol", "成交额": "amount_yuan",
    }).copy()
    out["trade_date"] = pd.to_datetime(out["trade_date"]).dt.strftime("%Y%m%d")
    out["ts_code"] = ts_code
    out["amount"] = pd.to_numeric(out["amount_yuan"], errors="coerce") / 1000.0
    return out


def compare_daily(left: pd.DataFrame, right: pd.DataFrame, label: str, left_close="close", right_close="close") -> dict:
    if left.empty or right.empty:
        return {"comparison": label, "matched_dates": 0, "max_close_abs_diff": np.nan,
                "max_vol_abs_diff": np.nan, "max_amount_abs_diff_thousand_yuan": np.nan,
                "status": "unavailable"}
    merged = left.merge(right, on="trade_date", suffixes=("_ts", "_ak"))
    return {
        "comparison": label,
        "matched_dates": len(merged),
        "max_close_abs_diff": (merged[f"{left_close}_ts"] - merged[f"{right_close}_ak"]).abs().max(),
        "max_vol_abs_diff": (merged["vol_ts"] - merged["vol_ak"]).abs().max() if {"vol_ts", "vol_ak"}.issubset(merged.columns) else np.nan,
        "max_amount_abs_diff_thousand_yuan": (merged["amount_ts"] - merged["amount_ak"]).abs().max() if {"amount_ts", "amount_ak"}.issubset(merged.columns) else np.nan,
        "status": "compared",
    }


ts_stock_raw_rep = stock_daily_raw.loc[stock_daily_raw["ts_code"].eq(REPRESENTATIVE_STOCK)].copy()
ts_stock_qfq_rep = stock_daily_qfq_check.loc[
    stock_daily_qfq_check["ts_code"].eq(REPRESENTATIVE_STOCK),
    ["trade_date", "open_qfq", "high_qfq", "low_qfq", "close_qfq", "vol", "amount"],
].rename(columns={"open_qfq": "open", "high_qfq": "high", "low_qfq": "low", "close_qfq": "close"})
ak_stock_raw = normalize_ak_daily(akshare_results.get("stock_raw", pd.DataFrame()), REPRESENTATIVE_STOCK)
ak_stock_qfq = normalize_ak_daily(akshare_results.get("stock_qfq", pd.DataFrame()), REPRESENTATIVE_STOCK)
ak_etf_raw = normalize_ak_daily(akshare_results.get("etf_raw", pd.DataFrame()), REPRESENTATIVE_ETF)
ak_etf_qfq = normalize_ak_daily(akshare_results.get("etf_qfq", pd.DataFrame()), REPRESENTATIVE_ETF)

cross_source_checks = pd.DataFrame([
    compare_daily(ts_stock_raw_rep, ak_stock_raw, "招商银行不复权：Tushare vs AkShare"),
    compare_daily(ts_stock_qfq_rep, ak_stock_qfq, "招商银行前复权：Tushare公式 vs AkShare"),
    compare_daily(etf_daily_raw, ak_etf_raw, "510300 ETF不复权：Tushare vs AkShare"),
    compare_daily(
        etf_daily_qfq_check[["trade_date", "close_qfq", "vol", "amount"]].rename(columns={"close_qfq": "close"}),
        ak_etf_qfq,
        "510300 ETF前复权：Tushare公式 vs AkShare",
    ),
])

# 分红按实施后的除权日对齐：AkShare 每10股派息 ÷10 后与 Tushare 税前每股金额比较。
ak_div = akshare_results.get("dividend", pd.DataFrame()).copy()
if not ak_div.empty:
    ak_div_norm = ak_div.loc[ak_div["方案进度"].astype(str).str.contains("实施", na=False)].copy()
    ak_div_norm["ex_date"] = pd.to_datetime(ak_div_norm["除权除息日"], errors="coerce").dt.strftime("%Y%m%d")
    ak_div_norm["ak_cash_per_share_pre_tax"] = pd.to_numeric(
        ak_div_norm["现金分红-现金分红比例"], errors="coerce"
    ) / 10.0
    ts_div_norm = dividends_raw.loc[
        dividends_raw["ts_code"].eq(REPRESENTATIVE_STOCK) & dividends_raw["div_proc"].eq("实施"),
        ["ex_date", "cash_div_tax", "cash_div", "end_date"],
    ].dropna(subset=["ex_date"])
    dividend_cross_check = ts_div_norm.merge(
        ak_div_norm[["ex_date", "ak_cash_per_share_pre_tax", "报告期"]], on="ex_date", how="outer", indicator=True
    )
    dividend_cross_check["pretax_abs_diff"] = (
        dividend_cross_check["cash_div_tax"] - dividend_cross_check["ak_cash_per_share_pre_tax"]
    ).abs()
else:
    dividend_cross_check = pd.DataFrame()

save_dataset(cross_source_checks, "跨源数值校验.csv", keys=["comparison"])
save_dataset(dividend_cross_check, "分红跨源校验.csv", keys=["ex_date"])
display(cross_source_checks)
display(dividend_cross_check.head(12))

,comparison,matched_dates,max_close_abs_diff,max_vol_abs_diff,max_amount_abs_diff_thousand_yuan,status
0,招商银行不复权：Tushare vs AkShare,1101,0.000000,0.50,0.130,compared
1,招商银行前复权：Tushare公式 vs AkShare,1101,2.712122,0.50,0.130,compared
2,510300 ETF不复权：Tushare vs AkShare,1101,0.000000,0.56,0.092,compared
3,510300 ETF前复权：Tushare公式 vs AkShare,1101,0.061802,0.56,0.092,compared


,ex_date,cash_div_tax,cash_div,end_date,ak_cash_per_share_pre_tax,报告期,_merge,pretax_abs_diff
0,20030716,0.120,0.096,20021231,0.120,2002-12-31,both,0.000000e+00
1,20040511,0.092,0.074,20031231,0.092,2003-12-31,both,0.000000e+00
2,20050620,0.110,0.099,20041231,0.110,2004-12-31,both,1.387779e-17
3,20060224,0.000,0.000,20060223,NaN,NaN,left_only,NaN
4,20060616,0.080,0.072,20051231,0.080,2005-12-31,both,0.000000e+00
5,20060921,0.180,0.162,20060920,NaN,NaN,left_only,NaN
6,20070704,0.120,0.108,20061231,0.120,2006-12-31,both,0.000000e+00
7,20080728,0.280,0.252,20071231,0.280,2007-12-31,both,5.551115e-17
8,20090703,0.100,0.060,20081231,0.100,2008-12-31,both,0.000000e+00
9,20100701,0.210,0.189,20091231,0.210,2009-12-31,both,2.775558e-17


### 6. 字段字典、完整性、重复、合法性与历史修订

In [9]:
field_rows = [
    # dataset, field, meaning, unit, required, notes
    ("stock_basic", "ts_code", "带交易所后缀的证券代码", "字符串", True, "银行清单主键"),
    ("stock_basic", "symbol", "6位交易代码", "字符串", True, "跨源对照时常用"),
    ("stock_basic", "industry", "Tushare行业分类", "分类", True, "本实验以 industry=银行筛选；分类口径可能修订"),
    ("stock_basic", "list_date", "上市日期", "YYYYMMDD", True, "完整性检查的起点"),
    ("daily/fund_daily", "trade_date", "交易日期", "YYYYMMDD", True, "与交易日历核对"),
    ("daily/fund_daily", "open/high/low/close", "未复权开高低收", "元", True, "股票 daily 与 ETF fund_daily 字段同义"),
    ("daily/fund_daily", "pre_close", "除权后的昨日收盘参考价", "元", True, "不一定等于上一交易日原始 close"),
    ("daily/fund_daily", "change", "收盘价相对 pre_close 的涨跌额", "元", True, "close - pre_close"),
    ("daily/fund_daily", "pct_chg", "相对 pre_close 的涨跌幅", "%", True, "change/pre_close×100"),
    ("daily/fund_daily", "vol", "成交量", "手", True, "通常1手=100股/份；特殊品种应查交易规则"),
    ("daily/fund_daily", "amount", "成交额", "千元", True, "与AkShare的元比较时需÷1000"),
    ("adj_factor/fund_adj", "adj_factor", "复权因子", "无量纲", True, "前复权按最新因子归一化"),
    ("dividend", "end_date", "分红归属财务年度/报告期", "YYYYMMDD", True, "不是现金到账日期"),
    ("dividend", "ann_date", "分红预案公告日", "YYYYMMDD", False, "同一年度可有多个阶段记录"),
    ("dividend", "div_proc", "实施进度", "分类", True, "预案/股东大会通过/实施等"),
    ("dividend", "stk_div", "每股送转合计", "股/股", False, "stk_bo_rate + stk_co_rate"),
    ("dividend", "stk_bo_rate", "每股送股比例", "股/股", False, "0.1表示每股送0.1股"),
    ("dividend", "stk_co_rate", "每股转增比例", "股/股", False, "0.1表示每股转增0.1股"),
    ("dividend", "cash_div", "每股现金分红（税后）", "元/股", False, "不要再除以10"),
    ("dividend", "cash_div_tax", "每股现金分红（税前）", "元/股", False, "与公告中10派X比较时，公告X÷10"),
    ("dividend", "record_date", "股权登记日", "YYYYMMDD", False, "收盘在册取得权益"),
    ("dividend", "ex_date", "除权除息日", "YYYYMMDD", False, "当日买入不取得本次权益"),
    ("dividend", "pay_date", "派息日", "YYYYMMDD", False, "现金实际发放日"),
    ("dividend", "div_listdate", "红股上市日", "YYYYMMDD", False, "无送转时通常为空"),
    ("dividend", "imp_ann_date", "实施公告日", "YYYYMMDD", False, "分派实施细节披露日"),
    ("dividend", "base_date", "分派基准日", "YYYYMMDD", False, "历史记录可能缺失"),
    ("dividend", "base_share", "基准股本", "万股", False, "分派计算基数"),
]
field_dictionary = pd.DataFrame(
    field_rows, columns=["dataset", "field", "meaning", "unit", "required", "notes"]
)
save_dataset(field_dictionary, "数据字段说明.csv", keys=["dataset", "field"])
display(field_dictionary)

,dataset,field,meaning,unit,required,notes
0,stock_basic,ts_code,带交易所后缀的证券代码,字符串,True,银行清单主键
1,stock_basic,symbol,6位交易代码,字符串,True,跨源对照时常用
2,stock_basic,industry,Tushare行业分类,分类,True,本实验以 industry=银行筛选；分类口径可能修订
3,stock_basic,list_date,上市日期,YYYYMMDD,True,完整性检查的起点
4,daily/fund_daily,trade_date,交易日期,YYYYMMDD,True,与交易日历核对
5,daily/fund_daily,open/high/low/close,未复权开高低收,元,True,股票 daily 与 ETF fund_daily 字段同义
6,daily/fund_daily,pre_close,除权后的昨日收盘参考价,元,True,不一定等于上一交易日原始 close
7,daily/fund_daily,change,收盘价相对 pre_close 的涨跌额,元,True,close - pre_close
8,daily/fund_daily,pct_chg,相对 pre_close 的涨跌幅,%,True,change/pre_close×100
9,daily/fund_daily,vol,成交量,手,True,通常1手=100股/份；特殊品种应查交易规则


In [10]:
quality_rows: list[dict] = []


def add_quality(dataset, check, value, severity, interpretation):
    quality_rows.append({
        "dataset": dataset, "check": check, "value": value,
        "severity": severity, "interpretation": interpretation,
    })


def profile_table(name: str, frame: pd.DataFrame, keys: list[str], required: list[str]):
    add_quality(name, "row_count", len(frame), "info", "当前返回行数")
    exact_dups = int(frame.duplicated().sum())
    add_quality(name, "exact_duplicate_rows", exact_dups, "high" if exact_dups else "info", "精确重复应为0")
    if all(k in frame.columns for k in keys):
        key_dups = int(frame.duplicated(keys, keep=False).sum())
        add_quality(name, "duplicate_primary_keys", key_dups, "high" if key_dups else "info", f"候选键={keys}")
    missing_columns = sorted(set(required) - set(frame.columns))
    add_quality(name, "missing_required_columns", ",".join(missing_columns), "critical" if missing_columns else "info", "稳定字段契约")
    for col in required:
        if col in frame.columns:
            rate = float(frame[col].isna().mean()) if len(frame) else np.nan
            add_quality(name, f"null_rate:{col}", rate, "high" if rate > 0 else "info", "必需字段空值率")


profile_table("银行股清单", bank_list, ["ts_code"], ["ts_code", "symbol", "name", "industry", "list_date"])
profile_table("股票不复权日行情", stock_daily_raw, ["ts_code", "trade_date"], ["ts_code", "trade_date", "open", "high", "low", "close", "vol", "amount"])
profile_table("股票复权因子", stock_adj_factors, ["ts_code", "trade_date"], ["ts_code", "trade_date", "adj_factor"])
profile_table("ETF不复权日行情", etf_daily_raw, ["ts_code", "trade_date"], ["ts_code", "trade_date", "open", "high", "low", "close", "vol", "amount"])
profile_table("ETF复权因子", etf_adj_factors, ["ts_code", "trade_date"], ["ts_code", "trade_date", "adj_factor"])
# 分红同一年度有多个流程阶段是业务事实，不把 end_date 当唯一键。
profile_table("分红原始数据", dividends_raw, ["ts_code", "end_date", "ann_date", "div_proc", "imp_ann_date"], ["ts_code", "end_date", "div_proc"])

open_dates = set(trade_calendar.loc[trade_calendar["is_open"].eq(1), "cal_date"].astype(str))
listing_dates = stock_basic.set_index("ts_code")["list_date"].to_dict()

def calendar_gaps(frame: pd.DataFrame, code: str, label: str):
    actual = set(frame.loc[frame["ts_code"].eq(code), "trade_date"].astype(str))
    listed = str(listing_dates.get(code, START_DATE))
    expected = {d for d in open_dates if max(START_DATE, listed) <= d <= END_DATE}
    missing_all = sorted(expected - actual)
    # 当天尚未到日线常规入库时间（约15~16点）时，不把当天判为历史缺口。
    today = RUN_AT.strftime("%Y%m%d")
    pending_today = [d for d in missing_all if d == today and RUN_AT.hour < 17]
    missing = [d for d in missing_all if d not in pending_today]
    extra = sorted(actual - expected)
    add_quality(label, "expected_open_days", len(expected), "info", "来自SSE交易日历")
    add_quality(label, "pending_current_open_day", len(pending_today), "info", "当前交易日尚未到收盘入库时间，不计为缺失")
    add_quality(label, "missing_open_days", len(missing), "medium" if missing else "info", "已排除盘中未入库当天；其余可能是停牌或源缺失，需停复牌表再分类")
    add_quality(label, "missing_open_dates_sample", ",".join(missing[:20]), "medium" if missing else "info", "最多展示20日")
    add_quality(label, "unexpected_non_open_days", len(extra), "high" if extra else "info", ",".join(extra[:20]))
    if actual:
        add_quality(label, "latest_trade_date", max(actual), "info", f"距运行日 {(pd.Timestamp(END_DATE)-pd.Timestamp(max(actual))).days} 个自然日")

for code_, name_ in TARGET_STOCKS.items():
    calendar_gaps(stock_daily_raw, code_, f"股票日行情:{name_}")
calendar_gaps(etf_daily_raw, REPRESENTATIVE_ETF, "ETF日行情:510300")

def ohlc_checks(frame: pd.DataFrame, label: str):
    invalid_high = (frame["high"] < frame[["open", "close", "low"]].max(axis=1)).sum()
    invalid_low = (frame["low"] > frame[["open", "close", "high"]].min(axis=1)).sum()
    negative_flow = ((frame["vol"] < 0) | (frame["amount"] < 0)).sum()
    add_quality(label, "invalid_high_rows", int(invalid_high), "high" if invalid_high else "info", "high应不低于开收低")
    add_quality(label, "invalid_low_rows", int(invalid_low), "high" if invalid_low else "info", "low应不高于开收高")
    add_quality(label, "negative_vol_or_amount_rows", int(negative_flow), "high" if negative_flow else "info", "成交量额不应为负")

ohlc_checks(stock_daily_raw, "股票不复权日行情")
ohlc_checks(etf_daily_raw, "ETF不复权日行情")

# 复权覆盖与区间末端锚定检查。
stock_missing_factor = int(stock_daily_qfq_check["adj_factor"].isna().sum())
etf_missing_factor = int(etf_daily_qfq_check["adj_factor"].isna().sum())
add_quality("股票前复权", "missing_adj_factor_rows", stock_missing_factor, "critical" if stock_missing_factor else "info", "每个行情日都应有因子")
add_quality("ETF前复权", "missing_adj_factor_rows", etf_missing_factor, "critical" if etf_missing_factor else "info", "每个行情日都应有因子")
stock_anchor_diff = stock_daily_qfq_check.groupby("ts_code", group_keys=False).tail(1).eval("abs(close_qfq-close)").max()
etf_anchor_diff = etf_daily_qfq_check.groupby("ts_code", group_keys=False).tail(1).eval("abs(close_qfq-close)").max()
add_quality("股票前复权", "latest_close_anchor_abs_diff", float(stock_anchor_diff), "high" if stock_anchor_diff > 1e-8 else "info", "区间最新日qfq应等于raw")
add_quality("ETF前复权", "latest_close_anchor_abs_diff", float(etf_anchor_diff), "high" if etf_anchor_diff > 1e-8 else "info", "区间最新日qfq应等于raw")

implemented = dividends_raw.loc[dividends_raw["div_proc"].eq("实施")]
for date_col in ["record_date", "ex_date"]:
    missing_n = int(implemented[date_col].isna().sum())
    add_quality("已实施分红", f"missing:{date_col}", missing_n, "medium" if missing_n else "info", "已实施权益事件应有登记日和除权日")
implemented_cash = implemented.loc[pd.to_numeric(implemented["cash_div_tax"], errors="coerce").fillna(0).gt(0)]
missing_pay_n = int(implemented_cash["pay_date"].isna().sum())
add_quality("已实施现金分红", "missing:pay_date", missing_pay_n, "medium" if missing_pay_n else "info", "仅对税前现金分红大于0的记录要求派息日")
fiscal_stage_duplicates = int(dividends_raw.duplicated(["ts_code", "end_date"], keep=False).sum())
add_quality("分红原始数据", "rows_sharing_same_fiscal_period", fiscal_stage_duplicates, "info", "多阶段/修订记录是预期现象，不应按年度直接去重")

# Tushare SDK 源码使用同一前复权公式，但 pro_bar 1.4.25 在 pandas 3.x 下仍调用
# fillna(method='bfill')，因此本实验直接使用 daily + adj_factor，避免隐藏兼容补丁。
pro_bar_source = inspect.getsource(ts.pro_bar)
sdk_formula_detected = (
    "data[col] * data['adj_factor'] / float(fcts['adj_factor'][0])" in pro_bar_source
)
factor_changes = stock_adj_factors.sort_values(["ts_code", "trade_date"]).copy()
factor_changes["previous_factor"] = factor_changes.groupby("ts_code")["adj_factor"].shift()
factor_jumps = factor_changes.loc[
    factor_changes["previous_factor"].notna()
    & factor_changes["adj_factor"].ne(factor_changes["previous_factor"]),
    ["ts_code", "trade_date", "previous_factor", "adj_factor"],
]
implemented_in_window = implemented.loc[
    implemented["ex_date"].between(START_DATE, END_DATE, inclusive="both"),
    ["ts_code", "ex_date"],
].dropna().drop_duplicates()
jump_keys = set(zip(factor_jumps["ts_code"], factor_jumps["trade_date"]))
ex_keys = set(zip(implemented_in_window["ts_code"], implemented_in_window["ex_date"]))
qfq_formula_audit = pd.DataFrame([
    ["SDK源码公式识别", bool(sdk_formula_detected), "pass" if sdk_formula_detected else "fail", "pro_bar源码应使用 raw×factor/latest_factor"],
    ["股票最新日锚定最大绝对差", float(stock_anchor_diff), "pass" if stock_anchor_diff <= 1e-8 else "fail", "每只股票区间最新日qfq应等于raw"],
    ["ETF最新日锚定最大绝对差", float(etf_anchor_diff), "pass" if etf_anchor_diff <= 1e-8 else "fail", "ETF区间最新日qfq应等于raw"],
    ["股票行情缺失复权因子行", stock_missing_factor, "pass" if stock_missing_factor == 0 else "fail", "行情与因子按证券+交易日一对一"],
    ["实施除权日数量(区间内)", len(ex_keys), "info", "五只样本股票"],
    ["复权因子跳变日数量", len(jump_keys), "info", "五只样本股票"],
    ["实施除权日与因子跳变匹配数", len(ex_keys & jump_keys), "pass" if ex_keys <= jump_keys else "review", "未匹配需查配股/送转或源修订"],
    ["未匹配实施除权日", ",".join(f"{c}:{d}" for c, d in sorted(ex_keys-jump_keys)), "pass" if ex_keys <= jump_keys else "review", "应为空"],
], columns=["check", "value", "status", "interpretation"])
save_dataset(qfq_formula_audit, "前复权验证摘要.csv", keys=["check"])

# 跨源价格：原始价应近似一致；前复权价允许因因子口径不同而不同，但必须显式暴露。
for _, check_row in cross_source_checks.iterrows():
    diff = check_row["max_close_abs_diff"]
    is_cross_vendor_qfq = "AkShare" in check_row["comparison"] and "前复权" in check_row["comparison"]
    severity = "high" if is_cross_vendor_qfq and pd.notna(diff) and diff > 0.01 else "info"
    add_quality(
        "跨源行情校验",
        check_row["comparison"],
        diff,
        severity,
        "前复权因子口径不一致会改变历史价格；应固定供应商和因子版本" if is_cross_vendor_qfq else "原始价或同源公式应近似一致",
    )

missing_report = pd.DataFrame(quality_rows)
save_dataset(missing_report, "缺失数据报告.csv", keys=["dataset", "check"])
display(missing_report.loc[missing_report["severity"].isin(["critical", "high", "medium"])])

,dataset,check,value,severity,interpretation
105,跨源行情校验,招商银行前复权：Tushare公式 vs AkShare,2.712122,high,前复权因子口径不一致会改变历史价格；应固定供应商和因子版本
107,跨源行情校验,510300 ETF前复权：Tushare公式 vs AkShare,0.061802,high,前复权因子口径不一致会改变历史价格；应固定供应商和因子版本


In [11]:
source_comparison = pd.DataFrame([
    ["银行股清单", "Tushare stock_basic", "AkShare 东财行业板块", "当前样本可交叉核对", "行业分类会调整；生产中保存有效期与来源版本"],
    ["股票不复权日线", "Tushare daily", "AkShare stock_zh_a_hist(adjust='') / 交易所", "两者均可取；需数值对齐", "停牌日无行；最终权威核验用交易所或持牌行情"],
    ["股票前复权", "daily + adj_factor 自建", "AkShare qfq", "Tushare因子更适合审计", "pro_bar 1.4.25 与 pandas 3.x 不兼容；前复权历史会随权益事件修订，必须快照化"],
    ["股票分红", "Tushare dividend", "AkShare stock_fhps_detail_em / 公司实施公告", "AkShare可用但字段不全", "AkShare现金比例为每10股；缺pay_date、红股上市日、税前税后双字段"],
    ["ETF日线", "Tushare fund_daily", "AkShare fund_etf_hist_em / 交易所", "字段近似但接口不共用", "还需基金NAV/IOPV、份额规模、PCF补充ETF研究"],
    ["ETF复权", "Tushare fund_adj", "AkShare qfq", "需要独立基金复权因子", "不能误用股票adj_factor接口"],
    ["停牌/缺口归因", "Tushare suspend_d + trade_cal", "交易所公告", "本实验仅发现候选缺口", "没有停复牌表不能把交易日缺行直接判为数据丢失"],
    ["历史修订", "本地快照+哈希+旧版归档", "任何单次在线API均不足", "必须自行版本化", "首次运行只建基线；以后运行才可发现修订"],
    ["官方法律事实", "公司/交易所公告", "巨潮资讯", "必须作为最终依据", "分红方案状态、日期或金额冲突时以实施公告为准"],
], columns=["requirement", "primary_source", "comparison_or_supplement", "assessment", "gap_or_action"])
save_dataset(source_comparison, "数据源对照表.csv", keys=["requirement"])

revision_report = pd.DataFrame(revision_findings)
revision_report.to_csv(DATA_DIR / "历史修订报告.csv", index=False, encoding="utf-8-sig")

display(source_comparison)
display(revision_report[["dataset", "status", "previous_rows", "current_rows", "added_exact_rows", "removed_exact_rows", "changed_keys", "note"]])

,requirement,primary_source,comparison_or_supplement,assessment,gap_or_action
0,银行股清单,Tushare stock_basic,AkShare 东财行业板块,当前样本可交叉核对,行业分类会调整；生产中保存有效期与来源版本
1,股票不复权日线,Tushare daily,AkShare stock_zh_a_hist(adjust='') / 交易所,两者均可取；需数值对齐,停牌日无行；最终权威核验用交易所或持牌行情
2,股票前复权,daily + adj_factor 自建,AkShare qfq,Tushare因子更适合审计,pro_bar 1.4.25 与 pandas 3.x 不兼容；前复权历史会随权益事件修订，...
3,股票分红,Tushare dividend,AkShare stock_fhps_detail_em / 公司实施公告,AkShare可用但字段不全,AkShare现金比例为每10股；缺pay_date、红股上市日、税前税后双字段
4,ETF日线,Tushare fund_daily,AkShare fund_etf_hist_em / 交易所,字段近似但接口不共用,还需基金NAV/IOPV、份额规模、PCF补充ETF研究
5,ETF复权,Tushare fund_adj,AkShare qfq,需要独立基金复权因子,不能误用股票adj_factor接口
6,停牌/缺口归因,Tushare suspend_d + trade_cal,交易所公告,本实验仅发现候选缺口,没有停复牌表不能把交易日缺行直接判为数据丢失
7,历史修订,本地快照+哈希+旧版归档,任何单次在线API均不足,必须自行版本化,首次运行只建基线；以后运行才可发现修订
8,官方法律事实,公司/交易所公告,巨潮资讯,必须作为最终依据,分红方案状态、日期或金额冲突时以实施公告为准


,dataset,status,previous_rows,current_rows,added_exact_rows,removed_exact_rows,changed_keys,note
0,银行股清单.csv,unchanged,42,42,0,0,0,与上次本地快照完全一致
1,不复权日行情.csv,unchanged,5505,5505,0,0,0,与上次本地快照完全一致
2,股票复权因子.csv,unchanged,5510,5510,0,0,0,与上次本地快照完全一致
3,前复权校验行情.csv,unchanged,5505,5505,0,0,0,与上次本地快照完全一致
4,分红原始数据.csv,unchanged,251,251,0,0,0,与上次本地快照完全一致
5,ETF不复权日行情.csv,unchanged,1101,1101,0,0,0,与上次本地快照完全一致
6,ETF复权因子.csv,unchanged,1101,1101,0,0,0,与上次本地快照完全一致
7,ETF前复权校验行情.csv,unchanged,1101,1101,0,0,0,与上次本地快照完全一致
8,跨源数值校验.csv,unchanged,4,4,0,0,0,与上次本地快照完全一致
9,分红跨源校验.csv,unchanged,27,27,0,0,0,与上次本地快照完全一致


## Takeaways

In [12]:
medium_or_worse = missing_report[missing_report["severity"].isin(["critical", "high", "medium"])]
calendar_gap_rows = missing_report[missing_report["check"].eq("missing_open_days")]
total_calendar_gaps = pd.to_numeric(calendar_gap_rows["value"], errors="coerce").fillna(0).sum()
exact_dups = missing_report[missing_report["check"].eq("exact_duplicate_rows")]
total_exact_dups = pd.to_numeric(exact_dups["value"], errors="coerce").fillna(0).sum()
revised_files = int((revision_report["status"] == "revised").sum())
compared = cross_source_checks.loc[cross_source_checks["status"].eq("compared")]
max_raw_diff = compared.loc[compared["comparison"].str.contains("不复权"), "max_close_abs_diff"].max()

summary_md = f"""
### 本次运行结论（{RUN_AT.strftime('%Y-%m-%d %H:%M %Z')}）

- Tushare 返回 **{len(bank_list)}** 只当前上市银行股；与仓库 AkShare 缓存相比，仅一方拥有的代码数分别为 **{len(ts_bank_codes-ak_bank_codes)} / {len(ak_bank_codes-ts_bank_codes)}**。
- 五只样本股票共有 **{len(stock_daily_raw):,}** 条不复权日行情；510300 ETF 有 **{len(etf_daily_raw):,}** 条。相对交易日历共有 **{int(total_calendar_gaps)}** 个候选缺口，候选缺口仍需停复牌表分类。
- 日行情所有数据集共发现 **{int(total_exact_dups)}** 条精确重复；OHLC、负成交量额和候选键结果见《缺失数据报告》输出。
- 前复权由原始价与复权因子独立重建，区间最新日与不复权价锚定；AkShare 对照的最大不复权收盘价绝对差为 **{max_raw_diff if pd.notna(max_raw_diff) else '未取得'}** 元。
- Tushare 分红 `cash_div_tax` / `cash_div` 的单位是 **元/股（税前/税后）**；AkShare“现金分红比例”是 **每10股派X元**，比较时必须除以10。日期语义见上方字段表。
- 股票与 ETF 的 OHLCV 字段和单位基本一致，但 **接口不一致**：股票用 `daily + adj_factor`，ETF 用 `fund_daily + fund_adj`。
- 本次发现 **{revised_files}** 个文件相对上次快照发生修订。若均为 baseline，表示这是首次运行，不能据此声称历史从未修订。
- **AkShare 不能单独满足全部生产要求**：它可完成股票/ETF不复权和前复权行情、基础分红与板块清单，但仍需 Tushare/交易所/公司公告补充可审计复权因子、税前税后每股分红、派息日/红股上市日、停复牌归因、ETF NAV/IOPV/PCF，以及本地历史版本管理。
"""
display(Markdown(summary_md))

expected_outputs = [
    "银行股清单.csv", "不复权日行情.csv", "前复权校验行情.csv", "分红原始数据.csv",
    "数据字段说明.csv", "缺失数据报告.csv", "数据源对照表.csv", "历史修订报告.csv",
    "ETF不复权日行情.csv", "ETF前复权校验行情.csv", "前复权验证摘要.csv", "跨源数值校验.csv", "分红跨源校验.csv",
]
output_manifest = pd.DataFrame({
    "file": expected_outputs,
    "exists": [(DATA_DIR / f).exists() for f in expected_outputs],
    "bytes": [(DATA_DIR / f).stat().st_size if (DATA_DIR / f).exists() else 0 for f in expected_outputs],
})
display(output_manifest)
assert output_manifest["exists"].all(), "存在未生成的必需产物"


### 本次运行结论（2026-07-23 10:43 CST）

- Tushare 返回 **42** 只当前上市银行股；与仓库 AkShare 缓存相比，仅一方拥有的代码数分别为 **0 / 0**。
- 五只样本股票共有 **5,505** 条不复权日行情；510300 ETF 有 **1,101** 条。相对交易日历共有 **0** 个候选缺口，候选缺口仍需停复牌表分类。
- 日行情所有数据集共发现 **0** 条精确重复；OHLC、负成交量额和候选键结果见《缺失数据报告》输出。
- 前复权由原始价与复权因子独立重建，区间最新日与不复权价锚定；AkShare 对照的最大不复权收盘价绝对差为 **0.0** 元。
- Tushare 分红 `cash_div_tax` / `cash_div` 的单位是 **元/股（税前/税后）**；AkShare“现金分红比例”是 **每10股派X元**，比较时必须除以10。日期语义见上方字段表。
- 股票与 ETF 的 OHLCV 字段和单位基本一致，但 **接口不一致**：股票用 `daily + adj_factor`，ETF 用 `fund_daily + fund_adj`。
- 本次发现 **0** 个文件相对上次快照发生修订。若均为 baseline，表示这是首次运行，不能据此声称历史从未修订。
- **AkShare 不能单独满足全部生产要求**：它可完成股票/ETF不复权和前复权行情、基础分红与板块清单，但仍需 Tushare/交易所/公司公告补充可审计复权因子、税前税后每股分红、派息日/红股上市日、停复牌归因、ETF NAV/IOPV/PCF，以及本地历史版本管理。


,file,exists,bytes
0,银行股清单.csv,True,3023
1,不复权日行情.csv,True,439874
2,前复权校验行情.csv,True,1008246
3,分红原始数据.csv,True,24181
4,数据字段说明.csv,True,2382
5,缺失数据报告.csv,True,9140
6,数据源对照表.csv,True,1532
7,历史修订报告.csv,True,2645
8,ETF不复权日行情.csv,True,94810
9,ETF前复权校验行情.csv,True,203910
